In [19]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy

import sys
from collections import defaultdict

from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

In [20]:
sys.argv = [
    'notebook',  
    '--modelname', 'llava:34b',
    '--data', 'gully',
    '--data_path','/home/data/',
    "--data_samples", 'train.jsonl',
    "--subset", 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [21]:
parser = argparse.ArgumentParser(description="A script to run V-LLMs on different image classification datasets")

In [22]:
# set up the argument parser
parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--data_samples", type=str, required=True, help="Name of the samples to run on")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 sampels it unloades the model from the GPU to avoid carshing.")


args = parser.parse_args()

In [23]:
# Create a result folder
print(args.results_dir)
if not os.path.exists(args.results_dir):
    os.makedirs(args.results_dir)

/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline


In [24]:
# Load test set:
file_path = 'Testing_Set.json'
#file_path = 'Validation_Set.json'
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [25]:
# Load the images
directory = '/root/home/data_jpg'
tile_dict = defaultdict(list)

for filename in os.listdir(directory):
    if filename.endswith('.jpg'):
        parts = filename.split('_')
        tile_number = parts[4].split('.')[0]
        tile_dict[tile_number].append(os.path.join(directory,filename))

tile_dict = dict(tile_dict)

print("Total number of tiles: {}".format(len(tile_dict.keys())))

Total number of tiles: 900


In [26]:
for tile_number, file_list in tile_dict.items():
    tile_dict[tile_number] = sorted(file_list, key=lambda x: int(x.split('_')[3]))

In [27]:
tile_dict_val = {}
for tile_number, file_list in tile_dict.items():
    if tile_number in data_keys_test:
        #tile_dict[tile_number] = sorted(file_list, key=lambda x: int(x.split('_')[2]))
        tile_dict_val[tile_number] = file_list

print('Number of Retained Annotated GT Images: ', len(tile_dict_val.keys()))

Number of Retained Annotated GT Images:  311


In [5]:
# load model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-72B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-72B-Instruct")

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 10.47it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 79.14 GiB of which 41.44 MiB is free. Process 2275830 has 55.89 GiB memory in use. Process 2300055 has 23.19 GiB memory in use. Of the allocated memory 22.75 GiB is allocated by PyTorch, and 35.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
# Messages containing multiple images and a text query
messages = [
    {
        "role": "user",
        "content": [
        ],
    }
]

content_prompt = {"type": "text", "text": "Identify the similarities between these images."}
content_img_add = {"type": "image", "image": "file:///path/to/image1.jpg"}




messages[0]['content'].append(content_msg)
messages[0]['content']

[{'type': 'text', 'text': 'Identify the similarities between these images.'},
 {'type': 'image', 'image': 'file:///path/to/image1.jpg'}]

In [12]:
messages[0]['content']

[{'type': 'text', 'text': 'Identify the similarities between these images.'}]

In [16]:
content_msg["image"] = "file://path/to/image3.jpg"

In [17]:
content_msg

{'type': 'image', 'image': 'file://path/to/image3.jpg'}

In [ ]:
# Sample messages for batch inference
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "file:///path/to/image1.jpg"},
            {"type": "image", "image": "file:///path/to/image2.jpg"},
            {"type": "text", "text": "What are the common elements in these pictures?"},
        ],
    }
]
messages2 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who are you?"},
]
# Combine messages for batch processing
messagesall = [messages1, messages2]

In [42]:
# Perform the inference
data = copy.deepcopy(tile_dict_val)

messages = []
for key, item in tqdm(data.items()):
    #print(key)
    #print(item)
    
    # Grab the message temp
    message_temp = [
    {
            "role": "user",
            "content": [
            ],
        }
    ]

    # Fill the message temp with the prompt
    content_prompt = {"type": "text", "text": ""}
    questions = ["Given the six images of the exact same area, collected over a period of 10 years. Are there any ephemeral gully appearances by looking at all of them together?. Reason and conclude with only yes or no. "]
    content_prompt["text"] = questions[0]

    # Fill the message temp with the images
    for img in item:
        #print(img)
        content_img_add = {"type": "image", "image": ""}
        content_img_add["image"] = img
        message_temp[0]["content"].append(content_img_add)
        
    message_temp[0]["content"].append(content_prompt)
    messages.append(message_temp)

    print(f"Img Tile: {key}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:00<00:00, 133064.22it/s]

Img Tile: 100
Img Tile: 1004
Img Tile: 1005
Img Tile: 1007
Img Tile: 1010
Img Tile: 1015
Img Tile: 1017
Img Tile: 1018
Img Tile: 1020
Img Tile: 1021
Img Tile: 1031
Img Tile: 1032
Img Tile: 1035
Img Tile: 1038
Img Tile: 104
Img Tile: 1040
Img Tile: 1041
Img Tile: 1043
Img Tile: 1049
Img Tile: 105
Img Tile: 1050
Img Tile: 1051
Img Tile: 1053
Img Tile: 1062
Img Tile: 1064
Img Tile: 1065
Img Tile: 1067
Img Tile: 107
Img Tile: 1073
Img Tile: 1078
Img Tile: 1081
Img Tile: 1085
Img Tile: 1095
Img Tile: 1098
Img Tile: 1099
Img Tile: 1114
Img Tile: 1117
Img Tile: 1121
Img Tile: 1123
Img Tile: 1128
Img Tile: 1133
Img Tile: 1137
Img Tile: 1138
Img Tile: 1140
Img Tile: 1141
Img Tile: 1143
Img Tile: 117
Img Tile: 118
Img Tile: 123
Img Tile: 124
Img Tile: 125
Img Tile: 126
Img Tile: 128
Img Tile: 129
Img Tile: 131
Img Tile: 135
Img Tile: 248
Img Tile: 250
Img Tile: 251
Img Tile: 253
Img Tile: 266
Img Tile: 267
Img Tile: 272
Img Tile: 273
Img Tile: 287
Img Tile: 288
Img Tile: 289
Img Tile: 301
Img Ti

In [43]:
messages

[[{'role': 'user',
   'content': [{'type': 'image',
     'image': '/root/home/data_jpg/neg_rgb_0_tile_100.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_1_tile_100.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_2_tile_100.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_3_tile_100.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_4_tile_100.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_5_tile_100.jpg'},
    {'type': 'text',
     'text': 'Given the six images of the exact same area, collected over a period of 10 years. Are there any ephemeral gully appearances by looking at all of them together?. Reason and conclude with only yes or no. '}]}],
 [{'role': 'user',
   'content': [{'type': 'image',
     'image': '/root/home/data_jpg/neg_rgb_0_tile_1004.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_1_tile_1004.jpg'},
    {'type': 'image', 'image': '/root/home/data_jpg/neg_rgb_

In [ ]:
# Preparation for batch inference
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=texts,
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Batch Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_texts)